# Read-only CT porespace analysis and visualisation

This notebook analyses a porespace volume within the cylinder mask written by [notebook 02](02_ct_cylinder_crop_examples.ipynb). It does not alter either HDF5 input.

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np

working_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from within the basalt-processing project.")

source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from basalt_processing.paths import load_config, resolve_path
from basalt_processing.porosity import calculate_porosity

CONFIG_PATH = project_root / "config" / "basalt.example.toml"
config = load_config(CONFIG_PATH)
paths_config = config.get("paths", {})
porosity_config = config["porosity"]
data_root = resolve_path(paths_config["data_root"], config.get("_config_dir"))

PORESPACE_H5 = resolve_path(porosity_config["porespace_h5"], data_root)
CYLINDER_CROP_H5 = resolve_path(porosity_config["cylinder_crop_h5"], data_root)
PORESPACE_DATASET = porosity_config["porespace_dataset"]
MASK_DATASET = porosity_config["mask_dataset"]
RUN_PYVISTA = False

In [ ]:
with h5py.File(PORESPACE_H5, "r") as porespace_file, h5py.File(CYLINDER_CROP_H5, "r") as crop_file:
    porespace = porespace_file[PORESPACE_DATASET][:]
    cylinder_mask = crop_file[MASK_DATASET][:]
    voxel_size_zyx = np.asarray(crop_file["data"].attrs.get("voxel_size", (1.0, 1.0, 1.0)))

statistics = calculate_porosity(porespace, cylinder_mask)
print(f"Porespace fraction within cylinder: {statistics.total_fraction:.4%}")

## Per-Z porespace fraction

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(statistics.by_z)
ax.set(xlabel="Z slice index", ylabel="Porespace fraction within cylinder")
ax.grid(True)

## Selected-Z porespace and mask panels

In [ ]:
selected_z = np.unique(np.linspace(0, porespace.shape[0] - 1, num=3, dtype=int))
fig, axes = plt.subplots(len(selected_z), 2, figsize=(8, 4 * len(selected_z)), squeeze=False)
for row, z_index in enumerate(selected_z):
    axes[row, 0].imshow(porespace[z_index], cmap="magma")
    axes[row, 0].set(title=f"Porespace, Z={z_index}", axis_off=True)
    axes[row, 1].imshow(cylinder_mask[z_index], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set(title=f"Cylinder mask, Z={z_index}", axis_off=True)
fig.tight_layout()

## Optional PyVista display

Install the optional visualisation dependency with `uv sync --extra viz`, then set `RUN_PYVISTA = True`.

In [ ]:
if RUN_PYVISTA:
    import pyvista

    dz, dy, dx = voxel_size_zyx
    grid = pyvista.ImageData(
        dimensions=np.array(porespace.shape[::-1]) + 1, spacing=(dx, dy, dz)
    )
    grid.cell_data["porespace"] = porespace.transpose(2, 1, 0).ravel(order="F")
    plotter = pyvista.Plotter()
    plotter.add_volume(grid, scalars="porespace", opacity="sigmoid")
    plotter.show()